# Graficas de medidas con iperf

Estas medidas comparan el rendimiento de red obtenido con `iperf3`. En TCP se mide el bitrate alcanzado y las retransmisiones hacia M2 y M3. En UDP se comparan las pruebas a 10, 50 y 100 Mbit/s usando el bitrate recibido, el jitter y la perdida de datagramas. Las graficas generadas se guardan en esta misma carpeta.

In [ ]:
from pathlib import Path
import html
import re

CARPETA = Path.cwd()


def bitrate_mbits(valor, unidad):
    valor = float(valor)
    factores = {'Kbits': 1 / 1000, 'Mbits': 1, 'Gbits': 1000}
    return valor * factores[unidad]


def destino_desde_nombre(nombre):
    return re.search(r'_m(\d)', nombre).group(1).join(['M', ''])


def parse_tcp(path):
    texto = path.read_text(encoding='utf-8', errors='replace')
    destino = destino_desde_nombre(path.name)
    filas = []
    patron = re.compile(r'\]\s+([0-9.]+)-([0-9.]+)\s+sec\s+([0-9.]+)\s+([KMG]Bytes)\s+([0-9.]+)\s+([KMG]bits)/sec\s*(\d+)?\s*(sender|receiver)')
    for linea in texto.splitlines():
        m = patron.search(linea)
        if not m:
            continue
        filas.append({
            'archivo': path.name,
            'protocolo': 'TCP',
            'destino': destino,
            'rol': m.group(8),
            'bitrate_mbits': bitrate_mbits(m.group(5), m.group(6)),
            'retransmisiones': int(m.group(7)) if m.group(7) else None,
        })
    return filas


def parse_udp(path):
    texto = path.read_text(encoding='utf-8', errors='replace')
    destino = destino_desde_nombre(path.name)
    objetivo = int(re.search(r'_(\d+)mbit', path.name).group(1))
    filas = []
    patron = re.compile(r'\]\s+([0-9.]+)-([0-9.]+)\s+sec\s+([0-9.]+)\s+([KMG]Bytes)\s+([0-9.]+)\s+([KMG]bits)/sec\s+([0-9.]+)\s+ms\s+(\d+)/(\d+)\s+\(([0-9.]+)%\)\s+(sender|receiver)')
    for linea in texto.splitlines():
        m = patron.search(linea)
        if not m:
            continue
        filas.append({
            'archivo': path.name,
            'protocolo': 'UDP',
            'destino': destino,
            'objetivo_mbits': objetivo,
            'rol': m.group(11),
            'bitrate_mbits': bitrate_mbits(m.group(5), m.group(6)),
            'jitter_ms': float(m.group(7)),
            'perdidos': int(m.group(8)),
            'total': int(m.group(9)),
            'perdida_pct': float(m.group(10)),
        })
    return filas


tcp = []
udp = []
for archivo in sorted(CARPETA.glob('*.txt')):
    if '_tcp_' in archivo.name:
        tcp.extend(parse_tcp(archivo))
    elif '_udp_' in archivo.name:
        udp.extend(parse_udp(archivo))

print('TCP:')
for fila in sorted(tcp, key=lambda x: (x['destino'], x['rol'])):
    retr = '' if fila['retransmisiones'] is None else f", retransmisiones={fila['retransmisiones']}"
    print(f"{fila['destino']} {fila['rol']}: bitrate={fila['bitrate_mbits']:.1f} Mbit/s{retr}")

print('\nUDP receiver:')
for fila in sorted([f for f in udp if f['rol'] == 'receiver'], key=lambda x: (x['destino'], x['objetivo_mbits'])):
    print(f"{fila['destino']} objetivo={fila['objetivo_mbits']:3} Mbit/s: recibido={fila['bitrate_mbits']:.1f} Mbit/s, jitter={fila['jitter_ms']:.3f} ms, perdida={fila['perdida_pct']:.1f}%")


def color(destino):
    return '#2878b5' if destino == 'M2' else '#d95f02'


def svg_barras(nombre, titulo, filas, campo, etiqueta_y, etiqueta_x, maximo=None):
    filas = list(filas)
    valores = [f[campo] for f in filas if f[campo] is not None]
    if not valores:
        return
    ymax = maximo if maximo is not None else max(valores) * 1.15
    ymax = ymax or 1

    ancho, alto = 860, 500
    margen_izq, margen_der, margen_sup, margen_inf = 75, 35, 60, 95
    base = alto - margen_inf
    area_h = alto - margen_sup - margen_inf
    paso = (ancho - margen_izq - margen_der) / len(filas)
    barra = paso * 0.58

    partes = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{ancho}' height='{alto}' viewBox='0 0 {ancho} {alto}'>",
        "<rect width='100%' height='100%' fill='white'/>",
        f"<text x='{ancho/2}' y='30' text-anchor='middle' font-family='Arial' font-size='22' font-weight='700'>{html.escape(titulo)}</text>",
        f"<line x1='{margen_izq}' y1='{base}' x2='{ancho-margen_der}' y2='{base}' stroke='#333'/>",
        f"<line x1='{margen_izq}' y1='{margen_sup}' x2='{margen_izq}' y2='{base}' stroke='#333'/>",
        f"<text x='22' y='{margen_sup + area_h/2}' transform='rotate(-90 22 {margen_sup + area_h/2})' text-anchor='middle' font-family='Arial' font-size='14'>{html.escape(etiqueta_y)}</text>",
        f"<text x='{ancho/2}' y='{alto-18}' text-anchor='middle' font-family='Arial' font-size='14'>{html.escape(etiqueta_x)}</text>",
    ]
    for i in range(6):
        v = ymax * i / 5
        y = base - (v / ymax) * area_h
        partes.append(f"<line x1='{margen_izq}' y1='{y:.1f}' x2='{ancho-margen_der}' y2='{y:.1f}' stroke='#ddd'/>")
        partes.append(f"<text x='{margen_izq-10}' y='{y+4:.1f}' text-anchor='end' font-family='Arial' font-size='12'>{v:.1f}</text>")

    for idx, fila in enumerate(filas):
        valor = fila[campo]
        x = margen_izq + idx * paso + (paso - barra) / 2
        h = 0 if valor is None else (valor / ymax) * area_h
        y = base - h
        etiqueta = fila['etiqueta']
        partes.append(f"<rect x='{x:.1f}' y='{y:.1f}' width='{barra:.1f}' height='{h:.1f}' fill='{color(fila['destino'])}'/>")
        partes.append(f"<text x='{x + barra/2:.1f}' y='{y-5:.1f}' text-anchor='middle' font-family='Arial' font-size='11'>{valor:.1f}</text>")
        partes.append(f"<text x='{x + barra/2:.1f}' y='{base+18}' text-anchor='end' transform='rotate(-35 {x + barra/2:.1f} {base+18})' font-family='Arial' font-size='11'>{html.escape(etiqueta)}</text>")
    partes.append("<rect x='690' y='52' width='16' height='16' fill='#2878b5'/><text x='712' y='65' font-family='Arial' font-size='13'>M2</text>")
    partes.append("<rect x='690' y='76' width='16' height='16' fill='#d95f02'/><text x='712' y='89' font-family='Arial' font-size='13'>M3</text>")
    partes.append('</svg>')
    (CARPETA / nombre).write_text('\n'.join(partes), encoding='utf-8')


def svg_lineas_udp(nombre, titulo, campo, etiqueta_y, maximo=None):
    filas = sorted([f for f in udp if f['rol'] == 'receiver'], key=lambda x: (x['destino'], x['objetivo_mbits']))
    objetivos = sorted({f['objetivo_mbits'] for f in filas})
    destinos = sorted({f['destino'] for f in filas})
    valores = [f[campo] for f in filas if f[campo] is not None]
    if not valores:
        return
    ymax = maximo if maximo is not None else max(valores) * 1.15
    ymax = ymax or 1

    ancho, alto = 900, 520
    margen_izq, margen_der, margen_sup, margen_inf = 75, 45, 60, 75
    base = alto - margen_inf
    area_w = ancho - margen_izq - margen_der
    area_h = alto - margen_sup - margen_inf
    xmin, xmax = min(objetivos), max(objetivos)
    xrango = xmax - xmin or 1

    def x_pos(objetivo):
        return margen_izq + ((objetivo - xmin) / xrango) * area_w

    def y_pos(valor):
        return base - (valor / ymax) * area_h

    partes = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{ancho}' height='{alto}' viewBox='0 0 {ancho} {alto}'>",
        "<rect width='100%' height='100%' fill='white'/>",
        f"<text x='{ancho/2}' y='30' text-anchor='middle' font-family='Arial' font-size='22' font-weight='700'>{html.escape(titulo)}</text>",
        f"<line x1='{margen_izq}' y1='{base}' x2='{ancho-margen_der}' y2='{base}' stroke='#333'/>",
        f"<line x1='{margen_izq}' y1='{margen_sup}' x2='{margen_izq}' y2='{base}' stroke='#333'/>",
        f"<text x='{ancho/2}' y='{alto-20}' text-anchor='middle' font-family='Arial' font-size='14'>Bitrate UDP configurado (Mbit/s)</text>",
        f"<text x='22' y='{margen_sup + area_h/2}' transform='rotate(-90 22 {margen_sup + area_h/2})' text-anchor='middle' font-family='Arial' font-size='14'>{html.escape(etiqueta_y)}</text>",
    ]
    for i in range(6):
        v = ymax * i / 5
        y = y_pos(v)
        partes.append(f"<line x1='{margen_izq}' y1='{y:.1f}' x2='{ancho-margen_der}' y2='{y:.1f}' stroke='#ddd'/>")
        partes.append(f"<text x='{margen_izq-10}' y='{y+4:.1f}' text-anchor='end' font-family='Arial' font-size='12'>{v:.1f}</text>")
    for objetivo in objetivos:
        x = x_pos(objetivo)
        partes.append(f"<line x1='{x:.1f}' y1='{base}' x2='{x:.1f}' y2='{base+6}' stroke='#333'/>")
        partes.append(f"<text x='{x:.1f}' y='{base+24}' text-anchor='middle' font-family='Arial' font-size='12'>{objetivo}</text>")

    offset = {'M2': -6, 'M3': 6}
    for destino in destinos:
        puntos = []
        for fila in [f for f in filas if f['destino'] == destino]:
            puntos.append((x_pos(fila['objetivo_mbits']) + offset.get(destino, 0), y_pos(fila[campo]), fila[campo]))
        path = ' '.join(('M' if i == 0 else 'L') + f" {x:.1f} {y:.1f}" for i, (x, y, _) in enumerate(puntos))
        partes.append(f"<path d='{path}' fill='none' stroke='{color(destino)}' stroke-width='2.5'/>")
        for x, y, valor in puntos:
            partes.append(f"<circle cx='{x:.1f}' cy='{y:.1f}' r='4.5' fill='{color(destino)}'/>")
            partes.append(f"<text x='{x:.1f}' y='{y-9:.1f}' text-anchor='middle' font-family='Arial' font-size='11'>{valor:.1f}</text>")
    partes.append("<circle cx='720' cy='56' r='5' fill='#2878b5'/><text x='735' y='61' font-family='Arial' font-size='13'>M2</text>")
    partes.append("<circle cx='720' cy='80' r='5' fill='#d95f02'/><text x='735' y='85' font-family='Arial' font-size='13'>M3</text>")
    partes.append('</svg>')
    (CARPETA / nombre).write_text('\n'.join(partes), encoding='utf-8')


tcp_receiver = []
for f in sorted([x for x in tcp if x['rol'] == 'receiver'], key=lambda x: x['destino']):
    g = dict(f)
    g['etiqueta'] = f"{f['destino']} receiver"
    tcp_receiver.append(g)

tcp_sender = []
for f in sorted([x for x in tcp if x['rol'] == 'sender'], key=lambda x: x['destino']):
    g = dict(f)
    g['etiqueta'] = f"{f['destino']} sender"
    tcp_sender.append(g)

svg_barras('iperf_tcp_bitrate.svg', 'Bitrate TCP recibido', tcp_receiver, 'bitrate_mbits', 'Bitrate (Mbit/s)', 'Destino')
svg_barras('iperf_tcp_retransmisiones.svg', 'Retransmisiones TCP', tcp_sender, 'retransmisiones', 'Retransmisiones', 'Destino')
svg_lineas_udp('iperf_udp_bitrate_recibido.svg', 'Bitrate UDP recibido', 'bitrate_mbits', 'Bitrate recibido (Mbit/s)')
svg_lineas_udp('iperf_udp_jitter.svg', 'Jitter UDP recibido', 'jitter_ms', 'Jitter (ms)')
svg_lineas_udp('iperf_udp_perdida.svg', 'Perdida UDP recibida', 'perdida_pct', 'Perdida (%)', maximo=100)

print('\nGraficas guardadas:')
for nombre in ['iperf_tcp_bitrate.svg', 'iperf_tcp_retransmisiones.svg', 'iperf_udp_bitrate_recibido.svg', 'iperf_udp_jitter.svg', 'iperf_udp_perdida.svg']:
    print(CARPETA / nombre)